In [1]:
!pip install -q streamlit pypdf sentence-transformers scikit-learn numpy cohere langchain-text-splitters
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 67.0 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 3s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.1
npm notice To update run: npm install -g npm@12.0.1
npm notice
⠙

In [ ]:
%%writefile functions.py
import os
import pypdf
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import cohere

def load_and_chunk_document(file_path_or_file, chunk_size=300, overlap=50):
    text = ""
    if isinstance(file_path_or_file, str):
        file_name = file_path_or_file
        if file_name.lower().endswith(".pdf"):
            reader = pypdf.PdfReader(file_path_or_file)
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + "\n"
        else:
            with open(file_path_or_file, "r", encoding="utf-8") as f:
                text = f.read()
    else:
        file_name = file_path_or_file.name
        if file_name.lower().endswith(".pdf"):
            reader = pypdf.PdfReader(file_path_or_file)
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + "\n"
        else:
            text = file_path_or_file.getvalue().decode("utf-8")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )
    chunks = text_splitter.split_text(text)
    return chunks

def create_embeddings(chunks, model_name="all-MiniLM-L6-v2"):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, show_progress_bar=False)
    return embeddings.tolist()

def search_chunks(query, chunks, embeddings, k=3):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    top_k_indices = np.argsort(similarities)[::-1][:k]
    top_chunks = [chunks[i] for i in top_k_indices]
    return top_chunks

def generate_answer(query, context, api_key):
    co = cohere.ClientV2(api_key=api_key)
    context_str = "\n---\n".join(context) if isinstance(context, list) else context
    prompt = f"""You are a helpful assistant. Answer the user question based ONLY on the provided context below.

Context:
{context_str}
Question:
{query}
Answer:"""
    response = co.chat(
        model="command-a-03-2025",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.message.content[0].text

Writing functions.py


In [ ]:
%%writefile app.py
import streamlit as st
from functions import (
    load_and_chunk_document,
    create_embeddings,
    search_chunks,
    generate_answer
)

st.set_page_config(page_title="RAG Document QA System", layout="wide")
st.title("RAG Question-Answering System")

st.sidebar.header("Configuration")
cohere_api_key = st.sidebar.text_input(
    "Cohere API Key",
    type="password",
    help="Enter your Cohere API key for answer generation"
)

k_value = st.sidebar.slider("Number of Chunks (k)", min_value=1, max_value=5, value=3)

st.header("1. Upload Document")
uploaded_file = st.file_uploader("Upload a PDF or TXT file", type=["pdf", "txt"])

if uploaded_file is not None:
    with st.spinner("Processing document and generating embeddings..."):
        chunks = load_and_chunk_document(uploaded_file)
        embeddings = create_embeddings(chunks)

    st.success(f"Document processed successfully! Created {len(chunks)} text chunks.")

    st.header("2. Ask a Question")
    query = st.text_input("Enter your question about the document:")

    search_button = st.button("Search & Answer")

    if search_button and query:
        with st.spinner("Searching for relevant chunks..."):
            top_k_chunks = search_chunks(query, chunks, embeddings, k=k_value)

        st.subheader(" Top Relevant Chunks")
        for idx, chunk in enumerate(top_k_chunks, start=1):
            with st.expander(f"Chunk {idx}"):
                st.write(chunk)

        if cohere_api_key:
            with st.spinner("Generating answer with Cohere..."):
                try:
                    answer = generate_answer(query, top_k_chunks, cohere_api_key)
                    st.subheader("Generated Answer")
                    st.success(answer)
                except Exception as e:
                    st.error(f"Error generating answer from Cohere: {str(e)}")
        else:
            st.warning(" Please provide a Cohere API key in the sidebar to generate AI answers.")
else:
    st.info("Please upload a PDF or TXT document to begin.")

Writing app.py
